In [1]:
! pip install kagglehub --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: kagglehub
    Found existing installation: kagglehub 0.3.2
    Uninstalling kagglehub-0.3.2:
      Successfully uninstalled kagglehub-0.3.2


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "rajsahu2004/lacuna-malaria-detection-dataset"
    )

print("Path to dataset files:", path)

100%|██████████| 3.99G/3.99G [01:06<00:00, 64.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rajsahu2004/lacuna-malaria-detection-dataset/versions/1


In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from keras.utils import Sequence
import cv2

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import TensorBoard, CSVLogger

from tensorflow.keras.models import Model
from tensorflow.keras import layers

In [4]:
os.listdir(path)

['SampleSubmission.csv', 'images', 'Test.csv', 'Train.csv']

In [5]:
path_images = path + "/images"
path_train_csv = path + "/Train.csv"
path_test_csv = path + "/Test.csv"

In [6]:
data_train = pd.read_csv(path_train_csv)
data_test = pd.read_csv(path_test_csv)

In [7]:
scaler = MinMaxScaler()

# Fit the scaler on the columns you want to normalize
scaler.fit(data_train[['xmin', 'ymin', 'xmax', 'ymax']])

# Transform the columns using the fitted scaler
data_train[['xmin', 'ymin', 'xmax', 'ymax']] = scaler.transform(data_train[['xmin', 'ymin', 'xmax', 'ymax']])

In [8]:
data_train["class"] = data_train["class"].map({"Trophozoite": 0,
                                               "NEG": 1,
                                               "WBC":2
                                               }
                                              )
data_train['image_path'] = data_train['Image_ID'].apply(lambda x: os.path.join(path_images, x))
train_data, val_data = train_test_split(data_train, test_size=0.2, random_state=42, stratify=data_train['class'])

In [9]:
# @title Params Train
BATCH_SIZE = 128 # @param ["32","64","128"] {"type":"raw"}
BUFFER_SIZE = 1000 # @param ["1000","10000"] {"type":"raw"}
IMG_SIZE = 224 # @param ["128","160","180","224"] {"type":"raw"}
csv_file = 'training.csv' # @param

In [10]:
#@title Utility function to load and preprocess images
def load_image(image_path, target_size):
    """
    Loads and preprocesses an image from a given path.

    Parameters:
    -----------
    image_path : str
        Path to the image.
    target_size : tuple
        Target size to resize the image (height, width).

    Returns:
    --------
    image : np.array
        Preprocessed image ready for model input.
    """
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, target_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image

In [ ]:
class CustomDataGenerator(Sequence):
    def __init__(self, dataframe, batch_size, img_size,
                 shuffle=True, base_path='', **kwargs):
        """
        Initializes the data generator for both class and bbox predictions.

        Parameters:
        -----------
        dataframe : pd.DataFrame
            The input DataFrame containing image paths, classes, and bbox coordinates.
        batch_size : int
            Number of samples per batch.
        img_size : int
            Target size to resize the images.
        shuffle : bool, optional
            If True, shuffles the data at the end of each epoch.
        base_path: str, optional
            The base path where the images are stored.
            If not provided, it assumes the paths in 'image_path' are absolute.
        """
        super().__init__(**kwargs)
        self.dataframe = dataframe
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.base_path = base_path  # Store the base path
        self.indexes = np.arange(len(self.dataframe))
        self.on_epoch_end()

        # Validar que colunas necessárias estão presentes no DataFrame
        assert 'image_path' in dataframe.columns, "DataFrame must contain 'image_path' column."
        assert 'class' in dataframe.columns, "DataFrame must contain 'class' column."
        assert all(x in dataframe.columns for x in ['xmin', 'ymin', 'xmax', 'ymax']), \
            "DataFrame must contain 'xmin', 'ymin', 'xmax', and 'ymax' columns."

    def __len__(self):
        """Returns the number of batches per epoch."""
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        """
        Generate one batch of data.

        Returns:
        --------
        tuple : (images, (class_labels, bbox_labels))
        """
        # Obter os índices do batch
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self.__data_generation(indexes)

    def on_epoch_end(self):
        """Shuffle indexes after each epoch if shuffle is set to True."""
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, indexes):
        """
        Generates data containing a batch of images, class labels, and bbox labels.

        Parameters:
        -----------
        indexes : list of int
            List of sample indexes to generate a batch from.

        Returns:
        --------
        tuple : (images, (class_labels, bbox_labels))
        """
        # Prealocar memória para os dados do batch
        images = np.empty((self.batch_size, self.img_size, self.img_size, 3), dtype=np.float32)
        class_labels = np.empty((self.batch_size), dtype=np.int32)
        bbox_labels = np.empty((self.batch_size, 4), dtype=np.float32)  # bbox with 4 values: [x_min, y_min, x_max, y_max]

        for i, idx in enumerate(indexes):
            # Carregar o caminho da imagem
            img_path = self.dataframe['image_path'].iloc[idx]

            # Concatenar o caminho base com o relativo se base_path for fornecido
            if self.base_path:
                img_path = os.path.join(self.base_path, img_path)

            # Carregar e preprocessar a imagem
            images[i] = load_image(img_path, target_size=(self.img_size, self.img_size))

            # Obter o rótulo da classe
            class_labels[i] = int(self.dataframe['class'].iloc[idx])

            # Obter as coordenadas do bbox (xmin, ymin, xmax, ymax)
            bbox_labels[i] = self.dataframe[['xmin', 'ymin', 'xmax', 'ymax']].iloc[idx].astype(np.float32).values

        return images, (class_labels, bbox_labels)

In [12]:
train_generator = CustomDataGenerator(train_data, BATCH_SIZE, IMG_SIZE)
val_generator = CustomDataGenerator(val_data, BATCH_SIZE, IMG_SIZE, shuffle=False)

In [15]:
def convolution_clf(image_input):
    base_model = VGG16(weights='imagenet',
                       include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))

    for layer in base_model.layers:
        layer.trainable = False

    x = base_model(image_input)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)

    output_conv = layers.Dense(3, activation='softmax', name="clf_output")(x)

    return output_conv

def dense_bbox(features_input):
    # Flatten the feature map for the bounding box prediction
    x = layers.Flatten()(features_input)

    # First dense layer with 256 units and ReLU activation
    x = layers.Dense(256, activation="relu", name="Dense_1")(x)

    # Second dense layer with 128 units and ReLU activation
    x = layers.Dense(128, activation="relu", name="Dense_2")(x)

    # Third dense layer with 64 units and ReLU activation
    x = layers.Dense(64, activation="relu", name="Dense_3")(x)

    # Final dense layer with 32 units and ReLU activation
    x = layers.Dense(32, activation="relu", name="Dense_final")(x)

    # Bounding box regression output (4 values: x, y, width, height)
    output_dense = layers.Dense(4, activation="linear", name="bbox_output")(x)

    return output_dense

def final_model():
    # Input layer for the image
    image_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3),
                               name="Image")

    # Convolutional layers for classification
    clf_output = convolution_clf(image_input)

    # Extract features for bounding box prediction
    features = layers.Flatten()(image_input)
    bbox_output = dense_bbox(features)

    # Create and return the final model with one input and two outputs (clf and bbox)
    model = Model(inputs=image_input, outputs=[clf_output, bbox_output])

    return model

In [ ]:
model = final_model()

tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
model.summary()

In [18]:
#@title Training Configuration
LEARNING_RATE = 0.001
EPOCHS = 15

# Definindo as funções de perda
bbox_loss = tf.keras.losses.Huber()
clf_loss = tf.keras.losses.SparseCategoricalCrossentropy()

# Usando um único otimizador
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss={'clf_output': clf_loss, 'bbox_output': bbox_loss},
              metrics={'clf_output': 'accuracy', 'bbox_output': 'mae'})

In [ ]:
history = model.fit(train_generator,
                    epochs=EPOCHS,
                    validation_data=val_generator,
                    callbacks=[CSVLogger(csv_file, append=True)]
                    )

In [ ]:
pd.read_csv(csv_file).head()

In [ ]:
%tensorboard --logdir log_dir